In [ ]:
import tarfile
import subprocess
from pathlib import Path
import concurrent.futures
import os

In [ ]:
source_dir = Path("/new_emit_dataset_vol/just_hypercubes")
target_dir = Path("/new_emit_dataset_vol_temp")
os.makedirs(target_dir, exist_ok=True)

def pack_granule(granule_dir):
    out_tarfile_name_partial = granule_dir.name + ".tar.partial"
    out_tarfile_path_partial = target_dir / out_tarfile_name_partial
    
    out_tarfile_name = granule_dir.name + ".tar"
    out_tarfile_path = target_dir / out_tarfile_name
    
    incomplete_flag = out_tarfile_path.with_suffix(".incomplete")
    
    if out_tarfile_path.exists() and not incomplete_flag.exists():
        return out_tarfile_name  # already packed
    
    if incomplete_flag.exists():
        print(f"Incomplete tar file {incomplete_flag} found. Deleting and packing again {granule_dir}.")
        if out_tarfile_path.exists():
            out_tarfile_path.unlink()
    
    # Create the incomplete flag file
    incomplete_flag.touch()
    
    cmd = [
        "tar", 
        "-cf", 
        str(out_tarfile_path_partial), 
        "-C", 
        str(granule_dir.parent), 
        granule_dir.name
    ]
    subprocess.run(cmd, check=True)
    
    # Rename the partial tar file to the final name
    os.rename(out_tarfile_path_partial, out_tarfile_path)
    
    # Remove the incomplete flag file
    incomplete_flag.unlink()
    
    return out_tarfile_name

granules = [d for d in source_dir.glob("*") if d.is_dir()]
print(f"Found {len(granules)} granules. Starting parallel tar...")

# 2. Spin up the thread pool
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    
    # Submit all the granule directories to the pool
    futures = [executor.submit(pack_granule, g) for g in granules]
    
    # As each thread finishes, it yields its result here so you can track progress
    for i, future in enumerate(concurrent.futures.as_completed(futures), 1):
        finished_tar = future.result()
        print(f"[{i}/{len(granules)}] Packaged {finished_tar}")

print("All granules packaged successfully!")


KeyboardInterrupt: 